# FinSight: Example Usage

This notebook is a quick tour of what FinSight can do. I load some real transaction data, find the recurring payments, predict where the balance is heading, get a few money tips, and make some charts all using the package directly in Python.

Heads up: the charts get saved as image files instead of popping up in a window, so this runs fine anywhere.

## 1. Load the transactions

The dataset stores unsigned amounts plus a type column (credit/debit); the loader applies the correct sign automatically.

In [1]:
from finsight import load_transactions

transactions = load_transactions('../data/sample_transactions.csv')
print(f'Loaded {len(transactions)} transactions')
print('First:', transactions[0])
print('Last :', transactions[-1])
# Note the signs: debits are negative, credits positive.
salary = next(t for t in transactions if t.description == 'Salary Credit')
rent = next(t for t in transactions if t.description == 'Rent Payment')
print('Salary amount:', salary.amount)
print('Rent amount  :', rent.amount)

Loaded 477 transactions
First: Transaction(date=datetime.date(2024, 1, 1), description='Salary Credit', amount=2500.0, category='salary')
Last : Transaction(date=datetime.date(2024, 8, 30), description='Cinema City', amount=-9.5, category='entertainment')
Salary amount: 2500.0
Rent amount  : -650.0


## 2. Find the recurring payments



In [2]:
from finsight import detect_recurring_payments, summarise_recurring

recurring = detect_recurring_payments(transactions)
for p in recurring:
    kind = 'income ' if p.is_income else 'expense'
    print(f'{p.description:<24} {p.frequency.value:<9} {kind} '
          f'{p.typical_amount:>9.2f}  x{p.occurrences}')

print()
print(summarise_recurring(recurring))

Salary Credit            monthly   income    2500.00  x9
Rent Payment             monthly   expense   -650.00  x9
Telekom Internet         monthly   expense    -39.99  x8
Gym Membership           monthly   expense    -29.99  x9
Adobe Creative Cloud     monthly   expense    -24.99  x9
Mobile Plan              monthly   expense    -19.99  x9
Netflix Subscription     monthly   expense    -12.99  x9
Spotify Premium          monthly   expense     -9.99  x9
GitHub Pro               monthly   expense     -4.99  x9

{'monthly_income': 2500.0, 'monthly_expenses': 792.93, 'monthly_net': 1707.07}


## 3. Predict the future balance

Assumed a starting balance of 1000 before the first transaction.

In [3]:
from finsight import current_balance, forecast_balance

start_balance = 1000.0
print('Current balance:', current_balance(transactions, start_balance))

fc = forecast_balance(transactions, recurring, horizon_days=90,
                      starting_balance=start_balance)
print('Final projected balance:', fc.final_balance)
print('Lowest projected point :', fc.minimum_balance)
print('First negative date    :', fc.first_negative_date())

Current balance: 5658.26
Final projected balance: 5715.8
Lowest projected point : 5168.17
First negative date    : None


## 4. Get some recommendations

Based on everything above, FinSight gives a few plain-language tips about savings and spending.

In [4]:
from finsight import generate_recommendations

recs = generate_recommendations(transactions, recurring, fc,
                                starting_balance=start_balance)
for r in recs:
    print(r)
    print()

[*] Thin emergency buffer
    Your balance covers about 2.2 months of spending. A widely used rule of thumb is 3-6 months. Building toward 7746 would give more resilience.

[i] Healthy savings rate
    You are saving about 18% of your income (~585.17/month). If this sits idle in a current account, consider a higher-interest savings account or a low-cost index fund.

[i] Recurring cash flow summary
    Detected recurring income of 2500.00/month and recurring expenses of 792.93/month, leaving 1707.07/month before discretionary spending.



## 5. Do it all in one go

Instead of calling each step by hand, `analyse` runs the whole thing at once and prints a nice summary report.

In [5]:
from finsight import analyse

result = analyse(transactions, horizon_days=90,
                 starting_balance=start_balance)
print(result.report())

  FinSight -- Financial Analysis Report
Period analysed : 2024-01-01 to 2024-08-30
Transactions    : 477
Current balance : 5,658.26

----------------------------------------------------------------
Recurring payments
----------------------------------------------------------------
  Salary Credit                    monthly   income     2500.00  (x9)
  Rent Payment                     monthly   expense    -650.00  (x9)
  Telekom Internet                 monthly   expense     -39.99  (x8)
  Gym Membership                   monthly   expense     -29.99  (x9)
  Adobe Creative Cloud             monthly   expense     -24.99  (x9)
  Mobile Plan                      monthly   expense     -19.99  (x9)
  Netflix Subscription             monthly   expense     -12.99  (x9)
  Spotify Premium                  monthly   expense      -9.99  (x9)
  GitHub Pro                       monthly   expense      -4.99  (x9)

  Monthly recurring income  :    2500.00
  Monthly recurring expenses:     792.93
  Mon

## 6. Make the charts

Finally, let's create three charts (balance forecast, spending by category, and recurring payments) and save each one as a PNG in the `outputs/` folder.

In [6]:
from pathlib import Path

from finsight.visualize import (
    plot_balance_forecast,
    plot_recurring_payments,
    plot_spending_by_category,
)

out = Path('../outputs')
out.mkdir(exist_ok=True)
p1 = plot_balance_forecast(transactions, result.forecast,
                           out / 'balance_forecast.png',
                           starting_balance=start_balance)
p2 = plot_spending_by_category(transactions,
                               out / 'spending_by_category.png')
p3 = plot_recurring_payments(result.recurring,
                             out / 'recurring_payments.png')
print('Saved:', p1, p2, p3, sep='\n')

Saved:
../outputs/balance_forecast.png
../outputs/spending_by_category.png
../outputs/recurring_payments.png


The saved balance-forecast chart looks like this:

![Balance forecast](../outputs/balance_forecast.png)